# Deepfake Detection — Colab Training
Trains all three models on a T4/A100 GPU.
**Estimated runtime: ~10 min per model on T4 (~30 min total).**

**Before running:** Runtime → Change runtime type → T4 GPU

---
**Reconnect-safe:** Cells 1–3 are idempotent — re-running them after a disconnect is safe.
Cell 5 (training) uses `--skip-existing` so already-finished models are never retrained.

## 0. Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    # cuDNN auto-tuner — free ~10% speed-up for fixed input sizes
    torch.backends.cudnn.benchmark = True
    print('cuDNN benchmark: ON')
else:
    print('GPU NOT FOUND — go to Runtime → Change runtime type → T4 GPU')

## 1. Clone repo & install dependencies
Idempotent: skips clone if the folder already exists (safe to re-run after a disconnect).

In [ ]:
import os
if not os.path.isdir('DeepfakeDetection'):
    !git clone https://github.com/MuhammadFad/DeepfakeDetection.git
    print('Cloned.')
else:
    print('Repo already cloned — pulling latest...')
    !git -C DeepfakeDetection pull --ff-only
%cd DeepfakeDetection
!pip install timm opencv-python plotly scikit-learn tqdm requests -q
!pip install grad-cam --no-deps -q
print('Dependencies installed.')

## 2. Download Kaggle dataset
Get your API key: kaggle.com → Settings → API → **Create New Token** → upload `kaggle.json` when prompted.

Idempotent: skips download and unzip if the dataset folder already exists.

In [ ]:
import os
DATASET_ROOT = 'real_vs_fake/real-vs-fake'

if os.path.isdir(DATASET_ROOT):
    print(f'Dataset already present at {DATASET_ROOT} — skipping download.')
else:
    from google.colab import files as colab_files
    colab_files.upload()  # upload kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -q
    !unzip -q 140k-real-and-fake-faces.zip
    print('Dataset downloaded and extracted.')

!ls {DATASET_ROOT}/

## 3. Populate image folders (5k train / 1k val / 1k test per class)
Idempotent: skips copying for any split/class that already has the target number of files.

In [ ]:
import shutil
from pathlib import Path

SRC    = Path(DATASET_ROOT)
DST    = Path('images')
COUNTS = {'train': 5000, 'valid': 1000, 'test': 1000}
MAP    = {'valid': 'val'}

for src_split, n in COUNTS.items():
    dst_split = MAP.get(src_split, src_split)
    for cls in ('real', 'fake'):
        src_dir = SRC / src_split / cls
        dst_dir = DST / dst_split / cls
        dst_dir.mkdir(parents=True, exist_ok=True)
        existing = set(p.name for p in dst_dir.iterdir())
        to_copy  = [f for f in sorted(src_dir.iterdir())[:n] if f.name not in existing]
        for f in to_copy:
            shutil.copy(f, dst_dir / f.name)
        total = len(list(dst_dir.iterdir()))
        status = f'(copied {len(to_copy)})' if to_copy else '(already complete)'
        print(f'  {dst_split}/{cls}: {total} images {status}')

## 4. Configure for T4 GPU
Sets batch size to 32 (safe for T4 16 GB VRAM). Idempotent — reads current value first.

In [ ]:
config_path = 'src/config.py'
config_text = open(config_path).read()

import re
current = re.search(r'BATCH_SIZE_TRAIN\s*=\s*(\d+)', config_text)
current_val = int(current.group(1)) if current else None
print(f'Current BATCH_SIZE_TRAIN = {current_val}')

if current_val != 32:
    config_text = re.sub(r'(BATCH_SIZE_TRAIN\s*=\s*)\d+', r'\g<1>32', config_text)
    open(config_path, 'w').write(config_text)
    print('BATCH_SIZE_TRAIN set to 32.')
else:
    print('Already 32 — no change needed.')

## 5. Train all three models
`--skip-existing` means a model with a valid checkpoint is never retrained.
If Colab disconnects mid-model, just re-run this cell — finished models are skipped automatically.

In [ ]:
!python scripts/train.py --model xception --skip-existing

In [ ]:
!python scripts/train.py --model vit_small_patch16_224 --skip-existing

In [ ]:
!python scripts/train.py --model efficientnet_b4 --skip-existing

## 6. Download checkpoints
Place the downloaded `.pth` files in `checkpoints/` locally and push with Git LFS:
```bash
git lfs track "*.pth"
git add checkpoints/
git commit -m "Add trained checkpoints"
git push
```

In [ ]:
from google.colab import files as colab_files
import os

for name in ['xception_best.pth', 'vit_small_patch16_224_best.pth', 'efficientnet_b4_best.pth']:
    path = f'checkpoints/{name}'
    if os.path.exists(path):
        print(f'Downloading {name}...')
        colab_files.download(path)
    else:
        print(f'MISSING: {path} — training may have failed')

log_path = 'output/training_log.txt'
if os.path.exists(log_path):
    print('\nDownloading training log...')
    colab_files.download(log_path)
else:
    print('\nNo training log found.')